# CHAOS time record

Notebook to help understand/ illustrate the Time records involved in the data.

# Preamble

In [ ]:
# setting up file path for imports
import sys
import os
from pathlib import Path

cwd = Path(os.getcwd())
print(cwd)
PROJECT_ROOT = cwd.resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

# installing relevant libraries
import chaosmagpy as cp
import numpy as np
import matplotlib.pyplot as plt
import h5py

# Project paths / utilities
from src.msc_thesis.paths import *
from src.msc_thesis.synSetup import *
from src.msc_thesis.synUtils import *
from src.msc_thesis.synDMD import *

/home/elliott/projects/msc_thesis_project/Last_stretch/Data_prep
['/home/elliott/projects/msc_thesis_project', '/home/elliott/miniforge3/envs/msc_thesis/lib/python311.zip', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/lib-dynload', '', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/site-packages', '/home/elliott/projects/msc_thesis_project']
['/home/elliott/projects/msc_thesis_project', '/home/elliott/projects/msc_thesis_project', '/home/elliott/miniforge3/envs/msc_thesis/lib/python311.zip', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/lib-dynload', '', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/site-packages', '/home/elliott/projects/msc_thesis_project']


/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/site-packages/chaosmagpy/model_utils.py:795: UserWarning: Input coordinates include the poles.
  warnings.warn('Input coordinates include the poles.')


['/home/elliott/projects/msc_thesis_project', '/home/elliott/projects/msc_thesis_project', '/home/elliott/projects/msc_thesis_project', '/home/elliott/miniforge3/envs/msc_thesis/lib/python311.zip', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/lib-dynload', '', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/site-packages', '/home/elliott/projects/msc_thesis_project']
C has shape dimensions: (148, 440, 440)


## Scientific context

CHAOS-8 (including updates such as CHAOS-8.6) are geomagnetic models derived from satellite data.

This model describes the evolution of Earth's internal magnetic field through time.

- Field evolution is represented by a gauss-coefficient time series
- This gauss coefficient time series is parameterised by a B-spline representation, with 0.5 year knot spacing (=independent information sample rate)
- In the most recent iteration (CHAOS-8.6) this time series covers 29.5 years (1997.1 - 2026.6, decimal years) 

The model combines numerous different data sources, including observatory and satellite observations. This leads to variable record quality.

- Orsted (earliest satellite input) first contributed to CHAOS-8 in 03/1999
- Due to this, it is advised that CHAOS is best constrained post 03/1999

For simplicity and stability, this project investigates the period 01/2000-01/2026 to give a 26 year record.

In [ ]:
# defining 'full' time record from scratch (decimal year)
full_record_start = 1997.1
full_record_end = 2026.6
full_dt = 0.2
full_record_times = full_record_start + (np.arange(0, 148, 1) * full_dt)

# defining the windowed start and end dates
window_record_start = 2000.1
window_record_end = 2026.1



f = h5py.File((f'{CHAOS_RESOL_DIR}/CHAOS_Resol_1997_2026_0806_SV.h5'),"r")

time_points = np.asarray(f['tp'])

print(time_points)

f.close()


[1997.1 1997.3 1997.5 1997.7 1997.9 1998.1 1998.3 1998.5 1998.7 1998.9
 1999.1 1999.3 1999.5 1999.7 1999.9 2000.1 2000.3 2000.5 2000.7 2000.9
 2001.1 2001.3 2001.5 2001.7 2001.9 2002.1 2002.3 2002.5 2002.7 2002.9
 2003.1 2003.3 2003.5 2003.7 2003.9 2004.1 2004.3 2004.5 2004.7 2004.9
 2005.1 2005.3 2005.5 2005.7 2005.9 2006.1 2006.3 2006.5 2006.7 2006.9
 2007.1 2007.3 2007.5 2007.7 2007.9 2008.1 2008.3 2008.5 2008.7 2008.9
 2009.1 2009.3 2009.5 2009.7 2009.9 2010.1 2010.3 2010.5 2010.7 2010.9
 2011.1 2011.3 2011.5 2011.7 2011.9 2012.1 2012.3 2012.5 2012.7 2012.9
 2013.1 2013.3 2013.5 2013.7 2013.9 2014.1 2014.3 2014.5 2014.7 2014.9
 2015.1 2015.3 2015.5 2015.7 2015.9 2016.1 2016.3 2016.5 2016.7 2016.9
 2017.1 2017.3 2017.5 2017.7 2017.9 2018.1 2018.3 2018.5 2018.7 2018.9
 2019.1 2019.3 2019.5 2019.7 2019.9 2020.1 2020.3 2020.5 2020.7 2020.9
 2021.1 2021.3 2021.5 2021.7 2021.9 2022.1 2022.3 2022.5 2022.7 2022.9
 2023.1 2023.3 2023.5 2023.7 2023.9 2024.1 2024.3 2024.5 2024.7 2024.9
 2025.

## Resolution context

The project makes use of the Resolution matrix provided by DTU. This requires data input of the following format:

- Gauss coefficient (MF) time series
- 29.5 year span incl. 1997.1-2026.6
- 0.2 year gauss coefficient knot spacing

This gives 148 data points total.

## Implemenation context

The input to DMD is a synthetic record. Due to resolution application this takes the form of:

- Gauss coefficient (SV) time series
- 29.5 year span incl. 1997.1-2026.6
- 0.2 year gauss coefficient knot spacing
- 148 time sample points

Ideally this would then be windowed to:

- Gauss coefficient (SV) time series
- 25 year span incl. 2000.1-2025.1
- 0.5 year gauss coefficient knot spacing
- 53 time sample points

This would require reprojecting the total input into B-spline and resampling at 0.5 years. 

For simplicity, the choice instead was to downsample the record to:

- Gauss coefficient (SV) time series
- 26 year span incl. 2000.1-2026.1
- 0.4 year gauss coefficient knot spacing
- 66 time sample points